> **Multi-node workspace required:** This is a multi-node example. To run it,
> your Modal workspace must have multi-node enabled. Contact
> [support@modal.com](mailto:support@modal.com) to enable multi-node.

# Multi-node Qwen3.6-27B full-weight training

This tutorial runs full-weight GRPO on
[Qwen3.6-27B](https://huggingface.co/Qwen/Qwen3.6-27B), a
27B-parameter hybrid language model from Qwen, using
[slime](https://github.com/THUDM/slime) across **4 nodes
(32 H100 GPUs)**.

The `Qwen3_6_27b_Recipe` preset configures colocated training and
rollout workers, EAGLE speculative decoding, CPU-offloaded Adam,
and DeepScaler math reward verification.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
import importlib.util

# Skip if modal_training_gym is already importable (e.g. a local editable
# checkout) so your edits keep taking effect and the env stays synced.
if importlib.util.find_spec('modal_training_gym') is None:
    %uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
from modal_training_gym import (
    HuggingFaceDataset,
    Qwen3_6_27B,
    Qwen3_6_27b_Recipe,
    TrainConfig,
)

## Dataset

We use [DAPO-Math-17k](https://huggingface.co/datasets/zhuzilin/dapo-math-17k),
a collection of math competition problems with verifiable answers.
The `deepscaler` reward model checks whether the model's response
matches the reference answer.

In [ ]:
class MathDataset(HuggingFaceDataset):
    hf_repo = "zhuzilin/dapo-math-17k"
    input_column = ""
    output_column = ""
    input_key = "prompt"
    label_key = "label"
    output_format = "jsonl"
    apply_chat_template = True
    always_prepare = True

## Launch training

`TrainConfig.launch()` starts the Modal app **detached** and returns a
`TrainingRun` handle as soon as training is spawned. Detached means the run
survives this process: closing the notebook, dropping your connection, or
hitting Ctrl-C leaves the 4 nodes training on Modal.

The handle records the Modal app id and the `train` function-call id, so the
run can be waited on — or cancelled — from anywhere later.

In [ ]:
def build_training_config() -> TrainConfig:
    return TrainConfig(
        model=Qwen3_6_27B(),
        dataset=MathDataset(n_rows=10),
        recipe=Qwen3_6_27b_Recipe(),
    )

training_run = build_training_config()
run = training_run.launch()

print(f"Training run:  {run.training_run_id}")
print(f"Modal app:     {run.modal_app_url}")
print(f"Function call: {run.function_call_id}")

## Reattach later

A launched run is addressable by id from any process — the handle is
reconstructed from the persisted `function_call_id`, so you don't need to
keep this session open:

```python
from modal_training_gym import TrainingRun

run = TrainingRun.from_id("<training_run_id>")
train_result = run.result()  # block for the TrainResult

# Or stop it early:
run.function_call.cancel(terminate_containers=True)
```

## Wait for the result (optional)

`run.result()` blocks until training finishes and returns the
`TrainResult`. Interrupting the wait does **not** stop the run: a launched
run always lives in a detached Modal app, so Ctrl-C here only stops
waiting. To actually stop training, cancel the call:
`run.function_call.cancel(terminate_containers=True)`.

Skip this cell entirely if you just want to launch and walk away.

In [ ]:
train_result = run.result()
print(f"Training complete: {train_result.training_run_id}")